<a href="https://colab.research.google.com/github/Deva2013/airline-disruption-management-system/blob/main/Airline_Disruption_Phase1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ── Environment Setup ──────────────────────────────────────────────────────
import os
import time
import zipfile
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

# Install Hugging Face Hub client
!pip install -q huggingface_hub

from huggingface_hub import HfApi, login

# ── Authenticate with Hugging Face using the Colab secret ──────────────────
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

HF_REPO_ID = "Dev123Hug456Face/airline-disruption-data"
hf_api = HfApi()

# ── Directory structure ──────────────────────────────────────────────────
# Everything lives on LOCAL Colab disk (/content) — fast, reliable, no
# Drive quota issues. This does NOT persist across sessions; that's
# expected. Finished "gold" outputs get pushed to Hugging Face Hub at the
# end of each stage instead of being written to Drive.

BASE_DIR       = Path('/content/airline-disruption')
DIR_RAW        = BASE_DIR / 'data' / 'raw'
DIR_PROCESSED  = BASE_DIR / 'data' / 'processed'
DIR_WEATHER    = BASE_DIR / 'data' / 'weather'

for d in [DIR_RAW, DIR_PROCESSED, DIR_WEATHER]:
    d.mkdir(parents=True, exist_ok=True)

print('Environment ready.')
print(f'Base directory: {BASE_DIR}')
print(f'HF repo target: {HF_REPO_ID}')

Environment ready.
Base directory: /content/airline-disruption
HF repo target: Dev123Hug456Face/airline-disruption-data


In [2]:
# ── Pull the verified BTS dataset from Hugging Face Hub ────────────────────
from huggingface_hub import hf_hub_download

bts_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename="bts_cleaned.parquet",
    repo_type="dataset",
    local_dir=DIR_PROCESSED,
)

print(f"Downloaded to: {bts_path}")

# Quick verification
df = pd.read_parquet(bts_path)
print(f"\nRows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Year range: {df['Year'].min()} - {df['Year'].max()}")
print(f"\nColumn list: {df.columns.tolist()}")

bts_cleaned.parquet: reconstructing file:   0%|          |  0.00B /  355MB            

bts_cleaned.parquet: downloading bytes:           |  0.00B            

Downloaded to: /content/airline-disruption/data/processed/bts_cleaned.parquet

Rows: 10,504,936
Columns: 41
Year range: 2022 - 2024

Column list: ['Year', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Airline', 'FlightNumber', 'OperatingAirline', 'OperatingAirlineCode', 'Origin', 'OriginCityName', 'OriginState', 'Dest', 'DestCityName', 'DestState', 'CRSDepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'TaxiOut', 'TaxiIn', 'CRSArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'Cancelled', 'CancellationCode', 'Diverted', 'AirTime', 'Distance', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay', 'CancellationReason', 'PrimaryDelayCause', 'SeverityTier', 'ScheduledDepHour', 'Season', 'IsWeekend']


In [3]:
# ── Dataset overview ────────────────────────────────────────────────────
total     = len(df)
cancelled = df['Cancelled'].sum()
dep_del   = df['DepDel15'].sum()
arr_del   = df['ArrDel15'].sum()

print('=' * 50)
print('  DATASET OVERVIEW')
print('=' * 50)
print(f'  Flights       : {total:,}')
print(f'  Date range    : {df["FlightDate"].min()} → {df["FlightDate"].max()}')
print(f'  Airlines      : {df["Airline"].nunique()}')
print(f'  Cancellation  : {cancelled/total*100:.2f}%  ({cancelled:,})')
print(f'  Dep delay≥15m : {dep_del/total*100:.2f}%  ({dep_del:,})')
print(f'  Arr delay≥15m : {arr_del/total*100:.2f}%  ({arr_del:,})')
print()
print('Severity breakdown:')
print(df['SeverityTier'].value_counts().to_string())

  DATASET OVERVIEW
  Flights       : 10,504,936
  Date range    : 2022-01-01 00:00:00 → 2024-12-31 00:00:00
  Airlines      : 10
  Cancellation  : 1.81%  (190,145)
  Dep delay≥15m : 20.04%  (2,105,664)
  Arr delay≥15m : 20.45%  (2,148,534)

Severity breakdown:
SeverityTier
On Time        8166257
Minor          1140970
Significant     697812
Severe          309752
Cancelled       190145


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Fetch Historical METAR Weather Data (IEM ASOS Archive)
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Pull hourly weather observations for all 10 hub airports,
#          2022-2024, from the Iowa Environmental Mesonet (IEM) ASOS
#          archive — a true historical data source (unlike
#          aviationweather.gov, which only serves the last 15 days).
#
# Storage: Raw per-airport-year CSVs are cached on LOCAL Colab disk
#          (BASE_DIR/data/metar_raw), NOT Google Drive. This data is
#          disposable/re-fetchable, so it doesn't need to persist
#          beyond this session.
# ═══════════════════════════════════════════════════════════════════════

import io

IEM_URL = 'https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py'
METAR_RAW = BASE_DIR / 'data' / 'metar_raw'
METAR_RAW.mkdir(exist_ok=True)

# Same 10 hub airports used in the BTS dataset, mapped to ICAO codes
# (ICAO codes are the 4-letter identifiers IEM's station data references)
AIRPORT_ICAO = {
    'JFK': 'KJFK', 'ORD': 'KORD', 'ATL': 'KATL', 'LAX': 'KLAX', 'DFW': 'KDFW',
    'SFO': 'KSFO', 'EWR': 'KEWR', 'MIA': 'KMIA', 'SEA': 'KSEA', 'BOS': 'KBOS',
}
START_YEAR, END_YEAR = 2022, 2024


def iem_station(icao):
    """IEM station IDs drop the leading 'K' for CONUS airports (KJFK -> JFK)."""
    return icao[1:] if icao.startswith('K') and len(icao) == 4 else icao


def fetch_iem_year(icao, year, max_retries=3):
    """
    Fetch one full year of METAR data for one airport from IEM.
    Retries with backoff on failure, and validates the response is
    real CSV data (not an HTML error/rate-limit page) before returning it.
    """
    params = [
        ('station', iem_station(icao)),
        ('data', 'all'),
        ('tz', 'Etc/UTC'),
        ('format', 'comma'),
        ('latlon', 'no'),
        ('sts', f'{year}-01-01T00:00:00Z'),
        ('ets', f'{year+1}-01-01T00:00:00Z' if year < END_YEAR else f'{year}-12-31T23:59:59Z'),
    ]
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(
                IEM_URL, params=params,
                headers={'User-Agent': 'airline-disruption-research (ASU student project)'},
                timeout=90
            )
            r.raise_for_status()
            text = r.text

            # Validation: IEM prefixes real responses with a few "#DEBUG:"
            # comment lines before the actual CSV header. Check the header
            # shows up near the top rather than requiring it be line 1 —
            # this rejects bad/empty responses instead of crashing later.
            if 'station,valid' in text[:600]:
                return text
            else:
                print(f'    [warn] {icao} {year}: unexpected response '
                      f'(attempt {attempt}), first 80 chars: {text[:80]!r}')
        except Exception as e:
            print(f'    [warn] {icao} {year}: {e} (attempt {attempt})')
        time.sleep(3 * attempt)  # back off longer with each retry
    return None


# ── Main fetch loop: one request per airport per year (30 total) ──────────
print('Fetching METAR data from IEM ASOS archive (local disk cache)...')
all_frames = []
failed = []

for iata, icao in AIRPORT_ICAO.items():
    print(f'  {iata} ({icao})')
    for year in range(START_YEAR, END_YEAR + 1):
        cache_file = METAR_RAW / f'{icao}_{year}.csv'

        # Reuse cached file if it exists and looks valid; otherwise fetch fresh
        if cache_file.exists():
            text = cache_file.read_text()
            if 'station,valid' not in text[:600]:
                cache_file.unlink()  # discard invalid cached file
                text = None
        else:
            text = None

        if text is None:
            text = fetch_iem_year(icao, year)
            if text:
                cache_file.write_text(text)
            time.sleep(1.5)  # be polite to IEM's server between requests

        if not text:
            failed.append((icao, year))
            continue

        # comment='#' skips IEM's "#DEBUG:" preamble lines automatically
        df_year = pd.read_csv(io.StringIO(text), na_values=['M'], comment='#')
        df_year['IATA'] = iata
        all_frames.append(df_year)

# ── Combine all airport-years into one dataframe ───────────────────────────
wx_raw = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()
print(f'\nTotal METAR records: {len(wx_raw):,}')
if failed:
    print(f'⚠️  Failed after retries: {failed}')
wx_raw.head(2)

Fetching METAR data from IEM ASOS archive (local disk cache)...
  JFK (KJFK)
  ORD (KORD)
    [warn] KORD 2023: 503 Server Error: Service Unavailable for url: https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?station=ORD&data=all&tz=Etc%2FUTC&format=comma&latlon=no&sts=2023-01-01T00%3A00%3A00Z&ets=2024-01-01T00%3A00%3A00Z (attempt 1)
    [warn] KORD 2023: 503 Server Error: Service Unavailable for url: https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?station=ORD&data=all&tz=Etc%2FUTC&format=comma&latlon=no&sts=2023-01-01T00%3A00%3A00Z&ets=2024-01-01T00%3A00%3A00Z (attempt 2)
  ATL (KATL)
  LAX (KLAX)


/tmp/ipykernel_1177/1516899102.py:105: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df_year = pd.read_csv(io.StringIO(text), na_values=['M'], comment='#')


  DFW (KDFW)
  SFO (KSFO)
  EWR (KEWR)
  MIA (KMIA)


/tmp/ipykernel_1177/1516899102.py:105: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df_year = pd.read_csv(io.StringIO(text), na_values=['M'], comment='#')


  SEA (KSEA)
  BOS (KBOS)

Total METAR records: 3,381,319


,station,valid,tmpf,dwpf,relh,drct,sknt,p01i,alti,mslp,...,ice_accretion_1hr,ice_accretion_3hr,ice_accretion_6hr,peak_wind_gust,peak_wind_drct,peak_wind_time,feel,metar,snowdepth,IATA
0,JFK,2022-01-01 00:00,49.0,48.0,96.32,200.0,6.0,NaN,NaN,1014.9,...,NaN,NaN,NaN,NaN,NaN,NaN,46.0,METAR JFK 010000Z AUTO 20006KT BR 09/09 RMK AO...,NaN,JFK
1,JFK,2022-01-01 00:00,NaN,NaN,NaN,200.0,5.0,NaN,29.97,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KJFK 010000Z AUTO 20005KT 8SM BKN004 OVC014 09...,NaN,JFK
